# Day 21 - Data Quality Checks

**Topic:** Data Quality Checks  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** ฝึกสร้าง rule-based Data Quality checks ด้วย PySpark เพื่อแยก PASS / FAIL และสร้าง DQ summary ที่ตรวจสอบย้อนหลังได้

วันนี้เราจะต่อจาก Bronze / Silver pattern โดยใช้ข้อมูล Silver-like orders แล้วสร้าง Data Quality checks แบบพื้นฐาน เช่น required field, valid range, valid date, uniqueness และ referential check จากนั้นสรุปผลเป็น DQ result / DQ summary ที่ใช้ดูคุณภาพข้อมูลได้ชัดเจนใน Fabric Notebook

## Goal of the day

1. แยกให้ออกว่า **data cleansing** กับ **data quality check** ต่างกันอย่างไร
2. สร้าง rule-based DQ flags ด้วย PySpark ได้ เช่น required field, range, date, uniqueness และ referential check
3. รวมหลาย DQ rules ให้กลายเป็น `dq_status` และ `dq_issues` ต่อ record ได้
4. สร้าง DQ summary / DQ log แบบเบื้องต้นเพื่อให้ผลตรวจสอบข้อมูลมองเห็นได้
5. เริ่มระวังการ drop bad records ทิ้งโดยไม่มี log หรือ evidence

## Why it matters in production

ใน production pipeline ข้อมูลที่ผ่าน Silver cleansing แล้วอาจยังไม่พร้อมใช้เสมอ เพราะ cleansing ทำให้ข้อมูล standardized มากขึ้น แต่ไม่ได้แปลว่าข้อมูลถูกต้องตาม business expectation ทุกข้อ

Data Quality Checks สำคัญเพราะ:

- ช่วยจับ missing required fields ก่อนส่งต่อ downstream
- ช่วยตรวจ business rule เช่น amount ต้องมากกว่า 0 หรือ status ต้องอยู่ใน allowed list
- ช่วยตรวจ duplicate business key ก่อนเกิดปัญหาในการ merge / report
- ช่วยตรวจ referential integrity เช่น order ต้องอ้างถึง customer ที่มีอยู่จริง
- ช่วยทำให้ pipeline observable เพราะมี DQ summary / DQ log ให้ตรวจย้อนหลังได้

Production takeaway:

> DQ check ไม่ควรเป็น logic ที่ซ่อนอยู่ใน notebook อย่างเดียว แต่ควรสร้างผลลัพธ์ที่ตรวจสอบย้อนหลังได้ เช่น record-level status, issue code และ summary count

## Key concepts

| Concept | Meaning |
|---|---|
| Data Quality Check | การตรวจข้อมูลด้วย rule ที่กำหนดไว้ เช่น required field, valid range, uniqueness |
| Required field | column ที่ต้องมีค่า เช่น `order_id`, `customer_id`, `order_date` |
| Valid range | rule ที่เช็กว่าค่าอยู่ในช่วงที่ยอมรับได้ เช่น `amount > 0` |
| Valid date | rule ที่เช็กว่าวันที่ parse ได้และไม่ผิด format |
| Uniqueness check | rule ที่เช็กว่า business key ไม่ซ้ำ เช่น `order_id` ต้อง unique |
| Referential check | rule ที่เช็กว่า key จาก fact/source มีอยู่ใน master/reference table |
| DQ flag | boolean column ที่บอกว่า record ผิด rule ใด rule หนึ่งหรือไม่ |
| DQ status | สถานะรวมของ record เช่น `PASS` หรือ `FAIL` |
| DQ issues | list หรือ string ของ issue codes ที่ช่วยบอกว่า record fail เพราะ rule ไหน |
| DQ summary | สรุปจำนวน passed / failed records ต่อ run, batch หรือ rule |

## Concept explanation

### Data cleansing vs Data Quality checks

**Data cleansing** คือการทำให้ข้อมูลสะอาดขึ้น เช่น trim string, cast type, normalize status, parse date

**Data Quality checks** คือการตรวจว่าข้อมูลผ่าน rule ที่เราคาดหวังหรือไม่ เช่น:

- `customer_id` ต้องไม่ว่าง
- `amount` ต้องมากกว่า 0
- `order_date` ต้อง parse เป็น date ได้
- `order_id` ต้องไม่ซ้ำ
- `customer_id` ต้องมีอยู่ใน customer master

พูดง่าย ๆ:

> Cleansing = ทำข้อมูลให้เป็นรูปแบบที่ใช้งานได้  
> DQ Check = ตรวจว่าข้อมูลน่าเชื่อถือพอจะส่งต่อหรือไม่

### Record-level DQ vs Summary-level DQ

ในการทำ DQ ที่ดี ควรมี 2 มุมมอง:

1. **Record-level result**: แต่ละแถวผ่านหรือไม่ผ่าน rule ไหน
2. **Summary-level result**: ทั้ง batch มีทั้งหมดกี่ records, pass กี่ records, fail กี่ records, fail ด้วย rule ไหนบ้าง

ถ้ามีแค่ summary อย่างเดียว เราจะรู้ว่า fail กี่ record แต่ไม่รู้ว่า record ไหน fail  
ถ้ามีแค่ record-level อย่างเดียว เราจะ debug ได้ แต่ monitor batch ภาพรวมยาก

### DQ status ควรมาจาก issue flags

Pattern ง่าย ๆ คือสร้าง boolean flags ก่อน เช่น:

```python
has_missing_customer_id = F.col("customer_id").isNull()
has_invalid_amount = F.col("amount") <= 0
```

จากนั้นค่อยรวมเป็น `dq_issues` และ `dq_status`

```text
ถ้า dq_issues ว่าง   -> PASS
ถ้า dq_issues ไม่ว่าง -> FAIL
```

### วันนี้ยังไม่ทำ quarantine ลึก

วันนี้จะเน้นการสร้าง DQ result และ DQ summary ก่อน ส่วนการแยก bad records ไป quarantine / reject table แบบจริงจังจะต่อใน Day 22


## Mermaid diagram: Data Quality Check Flow

```mermaid
flowchart LR
    A[Silver-like Orders] --> B[Standardize input columns]
    B --> C[Apply DQ flags]
    C --> D[Build dq_issues]
    D --> E{dq_status}
    E -- PASS --> F[Ready records]
    E -- FAIL --> G[Failed records visible for review]
    D --> H[DQ summary by batch / rule]
    H --> I[Optional Lakehouse DQ tables]
```

Key idea:

> DQ check ที่ดีควรทำให้ทั้ง record-level issues และ summary-level metrics มองเห็นได้ ไม่ใช่แค่ filter ทิ้งเงียบ ๆ

## Setup / imports

In [28]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window

StatementMeta(, 29e3b4f8-e9a5-4ca9-8c69-0a6147e4b56c, 30, Finished, Available, Finished, False)

## Create mock data

Dataset หลักมี 2 ชุด:

1. `df_silver_orders_raw` — orders ที่หน้าตาคล้าย Silver candidate แต่ยังมีปัญหา DQ บางอย่าง
2. `df_customer_master` — customer reference table สำหรับทำ referential check

In [29]:
orders_data = [
    ("O001", "C001", "2026-05-01", 1200.50, "completed", "web", "batch_001"),
    ("O002", "C002", "2026-05-02", 0.00, "completed", "web", "batch_001"),
    ("O003", "C999", "2026-05-03", 500.00, "completed", "mobile", "batch_001"),
    ("O004", None, "2026-05-04", 800.00, "completed", "mobile", "batch_001"),
    ("O005", "C003", "bad-date", 300.00, "completed", "crm", "batch_001"),
    ("O006", "C004", "2026-05-05", -50.00, "completed", "crm", "batch_002"),
    ("O006", "C004", "2026-05-05", 900.00, "completed", "crm", "batch_002"),
    ("O007", "C005", "2026-05-06", 700.00, "unknown", "web", "batch_002"),
    ("O008", "C002", "2026-05-07", 450.00, "pending", "web", "batch_002"),
]

orders_schema = T.StructType([
    T.StructField("order_id", T.StringType(), True),
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("order_date_raw", T.StringType(), True),
    T.StructField("amount", T.DoubleType(), True),
    T.StructField("order_status", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
    T.StructField("batch_id", T.StringType(), True),
])

df_silver_orders_raw = spark.createDataFrame(orders_data, orders_schema)

df_silver_orders_raw.show(truncate=False)
df_silver_orders_raw.printSchema()
print("Raw order count:", df_silver_orders_raw.count())

StatementMeta(, 29e3b4f8-e9a5-4ca9-8c69-0a6147e4b56c, 31, Finished, Available, Finished, False)

+--------+-----------+--------------+------+------------+-------------+---------+
|order_id|customer_id|order_date_raw|amount|order_status|source_system|batch_id |
+--------+-----------+--------------+------+------------+-------------+---------+
|O001    |C001       |2026-05-01    |1200.5|completed   |web          |batch_001|
|O002    |C002       |2026-05-02    |0.0   |completed   |web          |batch_001|
|O003    |C999       |2026-05-03    |500.0 |completed   |mobile       |batch_001|
|O004    |NULL       |2026-05-04    |800.0 |completed   |mobile       |batch_001|
|O005    |C003       |bad-date      |300.0 |completed   |crm          |batch_001|
|O006    |C004       |2026-05-05    |-50.0 |completed   |crm          |batch_002|
|O006    |C004       |2026-05-05    |900.0 |completed   |crm          |batch_002|
|O007    |C005       |2026-05-06    |700.0 |unknown     |web          |batch_002|
|O008    |C002       |2026-05-07    |450.0 |pending     |web          |batch_002|
+--------+------

In [30]:
customers_data = [
    ("C001", "Alice", "active"),
    ("C002", "Bob", "active"),
    ("C003", "Charlie", "inactive"),
    ("C004", "Diana", "active"),
]

customers_schema = T.StructType([
    T.StructField("customer_id", T.StringType(), False),
    T.StructField("customer_name", T.StringType(), True),
    T.StructField("customer_status", T.StringType(), True),
])

df_customer_master = spark.createDataFrame(customers_data, customers_schema)

df_customer_master.show(truncate=False)

StatementMeta(, 29e3b4f8-e9a5-4ca9-8c69-0a6147e4b56c, 32, Finished, Available, Finished, False)

+-----------+-------------+---------------+
|customer_id|customer_name|customer_status|
+-----------+-------------+---------------+
|C001       |Alice        |active         |
|C002       |Bob          |active         |
|C003       |Charlie      |inactive       |
|C004       |Diana        |active         |
+-----------+-------------+---------------+



## PySpark code examples

ใน section นี้จะไล่จากการเตรียม input → สร้าง DQ flags → รวม `dq_status` → สรุปผล → เขียน table แบบ optional เพื่อให้เห็น pattern ที่ใช้ซ้ำได้ใน Lakehouse

### Example 1: Standardize input before DQ checks

ก่อนเช็ก DQ ควรทำ standardization เบื้องต้นก่อน เช่น trim key, lower status และ parse date เพื่อให้ rule ทำงานบนรูปแบบข้อมูลที่คาดเดาได้

In [31]:
df_dq_base = (
    df_silver_orders_raw
    .withColumn("order_id", F.trim(F.col("order_id")))
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("order_status", F.lower(F.trim(F.col("order_status"))))
    .withColumn("order_date", F.to_date(F.col("order_date_raw"), "yyyy-MM-dd"))
)

df_dq_base.select(
    "order_id",
    "customer_id",
    "order_date_raw",
    "order_date",
    "amount",
    "order_status",
    "batch_id",
).show(truncate=False)

StatementMeta(, 29e3b4f8-e9a5-4ca9-8c69-0a6147e4b56c, 33, Finished, Available, Finished, False)

+--------+-----------+--------------+----------+------+------------+---------+
|order_id|customer_id|order_date_raw|order_date|amount|order_status|batch_id |
+--------+-----------+--------------+----------+------+------------+---------+
|O001    |C001       |2026-05-01    |2026-05-01|1200.5|completed   |batch_001|
|O002    |C002       |2026-05-02    |2026-05-02|0.0   |completed   |batch_001|
|O003    |C999       |2026-05-03    |2026-05-03|500.0 |completed   |batch_001|
|O004    |NULL       |2026-05-04    |2026-05-04|800.0 |completed   |batch_001|
|O005    |C003       |bad-date      |NULL      |300.0 |completed   |batch_001|
|O006    |C004       |2026-05-05    |2026-05-05|-50.0 |completed   |batch_002|
|O006    |C004       |2026-05-05    |2026-05-05|900.0 |completed   |batch_002|
|O007    |C005       |2026-05-06    |2026-05-06|700.0 |unknown     |batch_002|
|O008    |C002       |2026-05-07    |2026-05-07|450.0 |pending     |batch_002|
+--------+-----------+--------------+----------+----

### Example 2: Required field and valid date checks

กลุ่มแรกคือ technical DQ checks ที่มักเจอตั้งแต่ต้น pipeline:

- `order_id` ต้องไม่ null / blank
- `customer_id` ต้องไม่ null / blank
- `order_date` ต้อง parse ได้จาก `order_date_raw`

In [32]:
df_required_checks = (
    df_dq_base
    .withColumn("has_missing_order_id", F.col("order_id").isNull() | (F.col("order_id") == ""))
    .withColumn("has_missing_customer_id", F.col("customer_id").isNull() | (F.col("customer_id") == ""))
    .withColumn(
        "has_invalid_order_date",
        F.col("order_date_raw").isNotNull() & F.col("order_date").isNull()
    )
)

df_required_checks.select(
    "order_id",
    "customer_id",
    "order_date_raw",
    "order_date",
    "has_missing_order_id",
    "has_missing_customer_id",
    "has_invalid_order_date",
).show(truncate=False)

StatementMeta(, 29e3b4f8-e9a5-4ca9-8c69-0a6147e4b56c, 34, Finished, Available, Finished, False)

+--------+-----------+--------------+----------+--------------------+-----------------------+----------------------+
|order_id|customer_id|order_date_raw|order_date|has_missing_order_id|has_missing_customer_id|has_invalid_order_date|
+--------+-----------+--------------+----------+--------------------+-----------------------+----------------------+
|O001    |C001       |2026-05-01    |2026-05-01|false               |false                  |false                 |
|O002    |C002       |2026-05-02    |2026-05-02|false               |false                  |false                 |
|O003    |C999       |2026-05-03    |2026-05-03|false               |false                  |false                 |
|O004    |NULL       |2026-05-04    |2026-05-04|false               |true                   |false                 |
|O005    |C003       |bad-date      |NULL      |false               |false                  |true                  |
|O006    |C004       |2026-05-05    |2026-05-05|false           

### Example 3: Business rule checks

กลุ่มต่อมาคือ business-oriented DQ checks แบบง่าย:

- `amount` ต้องมากกว่า 0 และไม่เกิน threshold ที่กำหนด
- `order_status` ต้องอยู่ใน allowed values

In [33]:
allowed_statuses = ["completed", "cancelled", "pending"]
maximum_order_amount = 10000.00

df_business_checks = (
    df_required_checks
    .withColumn(
        "has_invalid_amount",
        F.col("amount").isNull()
        | (F.col("amount") <= 0)
        | (F.col("amount") > maximum_order_amount)
    )
    .withColumn(
        "has_invalid_status",
        F.col("order_status").isNull()
        | (~F.col("order_status").isin(allowed_statuses))
    )
)

df_business_checks.select(
    "order_id",
    "amount",
    "order_status",
    "has_invalid_amount",
    "has_invalid_status",
).show(truncate=False)

StatementMeta(, 29e3b4f8-e9a5-4ca9-8c69-0a6147e4b56c, 35, Finished, Available, Finished, False)

+--------+------+------------+------------------+------------------+
|order_id|amount|order_status|has_invalid_amount|has_invalid_status|
+--------+------+------------+------------------+------------------+
|O001    |1200.5|completed   |false             |false             |
|O002    |0.0   |completed   |true              |false             |
|O003    |500.0 |completed   |false             |false             |
|O004    |800.0 |completed   |false             |false             |
|O005    |300.0 |completed   |false             |false             |
|O006    |-50.0 |completed   |true              |false             |
|O006    |900.0 |completed   |false             |false             |
|O007    |700.0 |unknown     |false             |true              |
|O008    |450.0 |pending     |false             |false             |
+--------+------+------------+------------------+------------------+



### Example 4: Uniqueness check using window function

ถ้า `order_id` เป็น business key ที่ควร unique เราสามารถใช้ window function เพื่อหา duplicate count ต่อ key ได้

วันนี้ mark ทั้งสอง rows ที่มี `order_id` ซ้ำเป็น failed record เพื่อให้เห็นปัญหาแบบ record-level

In [34]:
order_key_window = Window.partitionBy("order_id")  # No orderBy is needed because this check only counts total records per order_id, not row sequence.

df_uniqueness_checks = (
    df_business_checks
    .withColumn("order_id_record_count", F.count("*").over(order_key_window))
    .withColumn(
        "has_duplicate_order_id",
        F.col("order_id").isNotNull() & (F.col("order_id_record_count") > 1)
    )
)

df_uniqueness_checks.select(
    "order_id",
    "customer_id",
    "amount",
    "order_id_record_count",
    "has_duplicate_order_id",
).orderBy("order_id", "amount").show(truncate=False)

StatementMeta(, 29e3b4f8-e9a5-4ca9-8c69-0a6147e4b56c, 36, Finished, Available, Finished, False)

+--------+-----------+------+---------------------+----------------------+
|order_id|customer_id|amount|order_id_record_count|has_duplicate_order_id|
+--------+-----------+------+---------------------+----------------------+
|O001    |C001       |1200.5|1                    |false                 |
|O002    |C002       |0.0   |1                    |false                 |
|O003    |C999       |500.0 |1                    |false                 |
|O004    |NULL       |800.0 |1                    |false                 |
|O005    |C003       |300.0 |1                    |false                 |
|O006    |C004       |-50.0 |2                    |true                  |
|O006    |C004       |900.0 |2                    |true                  |
|O007    |C005       |700.0 |1                    |false                 |
|O008    |C002       |450.0 |1                    |false                 |
+--------+-----------+------+---------------------+----------------------+



### Example 5: Referential check with customer master

Referential check ใช้ตรวจว่า key ที่อยู่ใน source/fact มีอยู่ใน reference/master table จริงหรือไม่

ในตัวอย่างนี้ `customer_id = C999` และ `C005` ไม่มีใน `df_customer_master` จึงควรถูก flag เป็น unknown customer

In [35]:
df_customer_reference = df_customer_master.select(
    F.col("customer_id").alias("master_customer_id"),
    "customer_name",
    "customer_status",
)

df_referential_checks = (
    df_uniqueness_checks
    .join(
        df_customer_reference,
        df_uniqueness_checks.customer_id == df_customer_reference.master_customer_id,
        "left"
    )
    .withColumn(
        "has_unknown_customer",
        F.col("customer_id").isNotNull() & F.col("master_customer_id").isNull()
    )
)

df_referential_checks.select(
    "order_id",
    "customer_id",
    "master_customer_id",
    "customer_name",
    "has_unknown_customer",
).show(truncate=False)

StatementMeta(, 29e3b4f8-e9a5-4ca9-8c69-0a6147e4b56c, 37, Finished, Available, Finished, False)

+--------+-----------+------------------+-------------+--------------------+
|order_id|customer_id|master_customer_id|customer_name|has_unknown_customer|
+--------+-----------+------------------+-------------+--------------------+
|O001    |C001       |C001              |Alice        |false               |
|O002    |C002       |C002              |Bob          |false               |
|O003    |C999       |NULL              |NULL         |true                |
|O004    |NULL       |NULL              |NULL         |false               |
|O005    |C003       |C003              |Charlie      |false               |
|O006    |C004       |C004              |Diana        |false               |
|O006    |C004       |C004              |Diana        |false               |
|O007    |C005       |NULL              |NULL         |true                |
|O008    |C002       |C002              |Bob          |false               |
+--------+-----------+------------------+-------------+--------------------+

### Example 6: Build dq_issues and dq_status

หลังจากมี flags แล้ว ค่อยรวมเป็น issue codes เพื่อให้รู้ว่าแต่ละ record fail เพราะ rule ไหน

Pattern นี้ช่วยให้ downstream หรือคน debug เห็นเหตุผลชัดกว่าการมีแค่ `is_valid = false`

In [36]:
dq_issue_exprs = [
    F.when(F.col("has_missing_order_id"), F.lit("REQUIRED_ORDER_ID")),
    F.when(F.col("has_missing_customer_id"), F.lit("REQUIRED_CUSTOMER_ID")),
    F.when(F.col("has_invalid_order_date"), F.lit("VALID_ORDER_DATE")),
    F.when(F.col("has_invalid_amount"), F.lit("VALID_AMOUNT_RANGE")),
    F.when(F.col("has_invalid_status"), F.lit("VALID_ORDER_STATUS")),
    F.when(F.col("has_duplicate_order_id"), F.lit("UNIQUE_ORDER_ID")),
    F.when(F.col("has_unknown_customer"), F.lit("REFERENTIAL_CUSTOMER")),
]

run_id = "day21_run_001"

df_dq_result = (
    df_referential_checks
    .withColumn("dq_issues_raw", F.array(*dq_issue_exprs))  # * unpacks list into multiple DQ issue arguments
    .withColumn("dq_issues", F.expr("filter(dq_issues_raw, x -> x is not null)"))
    .drop("dq_issues_raw")
    .withColumn(
        "dq_status",
        F.when(F.size(F.col("dq_issues")) == 0, F.lit("PASS")).otherwise(F.lit("FAIL"))
    )
    .withColumn("run_id", F.lit(run_id))
    .withColumn("dq_checked_at", F.current_timestamp())
)

# Tips:
# - array(value1, value2, ...) combines multiple values into one array.
# - Higher-order function: a function that applies logic to elements inside an array.
#   Example: filter(array_column, element -> condition) keeps only array elements that match the condition.
# - Lambda expression / anonymous function: inline logic defined without a separate function name.
#   Example: element -> condition defines the filtering logic for each array element.

df_dq_result.select(
    "run_id",
    "batch_id",
    "order_id",
    "customer_id",
    "amount",
    "order_status",
    "dq_status",
    "dq_issues",
).orderBy("order_id", "amount").show(truncate=False)

StatementMeta(, 29e3b4f8-e9a5-4ca9-8c69-0a6147e4b56c, 38, Finished, Available, Finished, False)

+-------------+---------+--------+-----------+------+------------+---------+------------------------------------------+
|run_id       |batch_id |order_id|customer_id|amount|order_status|dq_status|dq_issues                                 |
+-------------+---------+--------+-----------+------+------------+---------+------------------------------------------+
|day21_run_001|batch_001|O001    |C001       |1200.5|completed   |PASS     |[]                                        |
|day21_run_001|batch_001|O002    |C002       |0.0   |completed   |FAIL     |[VALID_AMOUNT_RANGE]                      |
|day21_run_001|batch_001|O003    |C999       |500.0 |completed   |FAIL     |[REFERENTIAL_CUSTOMER]                    |
|day21_run_001|batch_001|O004    |NULL       |800.0 |completed   |FAIL     |[REQUIRED_CUSTOMER_ID]                    |
|day21_run_001|batch_001|O005    |C003       |300.0 |completed   |FAIL     |[VALID_ORDER_DATE]                        |
|day21_run_001|batch_002|O006    |C004  

### Example 7: Split ready records and failed records

วันนี้ยังไม่ลงลึก quarantine table แต่เราสามารถแยก DataFrame สำหรับ records ที่พร้อมส่งต่อ กับ records ที่ fail DQ เพื่อ review ได้ทันที

In [37]:
df_ready_orders = (
    df_dq_result
    .filter(F.col("dq_status") == "PASS")
    .select(
        "order_id",
        "customer_id",
        "order_date",
        "amount",
        "order_status",
        "source_system",
        "batch_id",
    )
)

df_failed_orders_visible = (
    df_dq_result
    .filter(F.col("dq_status") == "FAIL")
    .select(
        "order_id",
        "customer_id",
        "order_date_raw",
        "amount",
        "order_status",
        "batch_id",
        "dq_issues",
    )
)

print("Ready order count:", df_ready_orders.count())
print("Failed order count:", df_failed_orders_visible.count())

df_ready_orders.show(truncate=False)
df_failed_orders_visible.orderBy("order_id", "amount").show(truncate=False)

StatementMeta(, 29e3b4f8-e9a5-4ca9-8c69-0a6147e4b56c, 39, Finished, Available, Finished, False)

Ready order count: 2
Failed order count: 7
+--------+-----------+----------+------+------------+-------------+---------+
|order_id|customer_id|order_date|amount|order_status|source_system|batch_id |
+--------+-----------+----------+------+------------+-------------+---------+
|O001    |C001       |2026-05-01|1200.5|completed   |web          |batch_001|
|O008    |C002       |2026-05-07|450.0 |pending     |web          |batch_002|
+--------+-----------+----------+------+------------+-------------+---------+

+--------+-----------+--------------+------+------------+---------+------------------------------------------+
|order_id|customer_id|order_date_raw|amount|order_status|batch_id |dq_issues                                 |
+--------+-----------+--------------+------+------------+---------+------------------------------------------+
|O002    |C002       |2026-05-02    |0.0   |completed   |batch_001|[VALID_AMOUNT_RANGE]                      |
|O003    |C999       |2026-05-03    |500.0 |

### Example 8: Create DQ run summary

DQ summary ช่วยให้เห็นภาพรวมของ batch/run ว่าข้อมูลผ่านกี่ records และ fail กี่ records

วันนี้ใช้ summary แบบง่ายระดับ `run_id` + `batch_id`

In [46]:
df_dq_run_summary = (
    df_dq_result
    .groupBy("run_id", "batch_id")
    .agg(
        F.count("*").alias("total_records"),
        F.sum(F.when(F.col("dq_status") == "PASS", 1).otherwise(0)).alias("passed_records"),
        F.sum(F.when(F.col("dq_status") == "FAIL", 1).otherwise(0)).alias("failed_records"),
        F.sum(F.col("has_missing_customer_id").cast("int")).alias("missing_customer_id_records"),
        F.sum(F.col("has_invalid_amount").cast("int")).alias("invalid_amount_records"),
        F.sum(F.col("has_duplicate_order_id").cast("int")).alias("duplicate_order_id_records"),
        F.sum(F.col("has_unknown_customer").cast("int")).alias("unknown_customer_records"),
    )
    .withColumn(
        "pass_rate_percent",
        F.round(F.col("passed_records") * 100.0 / F.col("total_records"), 2)  # Use passed * 100 / total instead of passed / total * 100 to reduce Fabric Spark Advisor rounding warning.
    )
    .orderBy("batch_id")
)

df_dq_run_summary.show(truncate=False)

StatementMeta(, 29e3b4f8-e9a5-4ca9-8c69-0a6147e4b56c, 48, Finished, Available, Finished, False)

+-------------+---------+-------------+--------------+--------------+---------------------------+----------------------+--------------------------+------------------------+-----------------+
|run_id       |batch_id |total_records|passed_records|failed_records|missing_customer_id_records|invalid_amount_records|duplicate_order_id_records|unknown_customer_records|pass_rate_percent|
+-------------+---------+-------------+--------------+--------------+---------------------------+----------------------+--------------------------+------------------------+-----------------+
|day21_run_001|batch_001|5            |1             |4             |1                          |1                     |0                         |1                       |20.0             |
|day21_run_001|batch_002|4            |1             |3             |0                          |1                     |2                         |1                       |25.0             |
+-------------+---------+-------------+------

### Example 9: Create rule-level DQ summary

Rule-level summary ช่วยตอบคำถามว่า rule ไหน fail เยอะที่สุดใน batch นี้

Pattern นี้ใช้ `explode(dq_issues)` เพื่อเปลี่ยน array ของ issue codes ให้เป็นแถว แล้วค่อย aggregate ต่อ

In [47]:
df_dq_rule_summary = (
    df_dq_result
    .filter(F.size(F.col("dq_issues")) > 0)
    .select(
        "run_id",
        "batch_id",
        F.explode("dq_issues").alias("dq_rule_code")
    )
    .groupBy("run_id", "batch_id", "dq_rule_code")
    .agg(F.count("*").alias("failed_records"))
    .orderBy("batch_id", "dq_rule_code")
)

df_dq_rule_summary.show(truncate=False)

StatementMeta(, 29e3b4f8-e9a5-4ca9-8c69-0a6147e4b56c, 49, Finished, Available, Finished, False)

+-------------+---------+--------------------+--------------+
|run_id       |batch_id |dq_rule_code        |failed_records|
+-------------+---------+--------------------+--------------+
|day21_run_001|batch_001|REFERENTIAL_CUSTOMER|1             |
|day21_run_001|batch_001|REQUIRED_CUSTOMER_ID|1             |
|day21_run_001|batch_001|VALID_AMOUNT_RANGE  |1             |
|day21_run_001|batch_001|VALID_ORDER_DATE    |1             |
|day21_run_001|batch_002|REFERENTIAL_CUSTOMER|1             |
|day21_run_001|batch_002|UNIQUE_ORDER_ID     |2             |
|day21_run_001|batch_002|VALID_AMOUNT_RANGE  |1             |
|day21_run_001|batch_002|VALID_ORDER_STATUS  |1             |
+-------------+---------+--------------------+--------------+



### Example 10: Write DQ result and summary to Lakehouse tables

In [48]:
dq_result_table_name = "day21_dq_order_result"
dq_run_summary_table_name = "day21_dq_run_summary"
dq_rule_summary_table_name = "day21_dq_rule_summary"

(
    df_dq_result
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(dq_result_table_name)
)

(
    df_dq_run_summary
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(dq_run_summary_table_name)
)

(
    df_dq_rule_summary
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(dq_rule_summary_table_name)
)

print("Written tables:")
print("-", dq_result_table_name)
print("-", dq_run_summary_table_name)
print("-", dq_rule_summary_table_name)

StatementMeta(, 29e3b4f8-e9a5-4ca9-8c69-0a6147e4b56c, 50, Finished, Available, Finished, False)

Written tables:
- day21_dq_order_result
- day21_dq_run_summary
- day21_dq_rule_summary


### Example 11: Read DQ summary back from Lakehouse

In [49]:
spark.read.table(dq_run_summary_table_name).show(truncate=False)
spark.read.table(dq_rule_summary_table_name).show(truncate=False)

StatementMeta(, 29e3b4f8-e9a5-4ca9-8c69-0a6147e4b56c, 51, Finished, Available, Finished, False)

+-------------+---------+-------------+--------------+--------------+---------------------------+----------------------+--------------------------+------------------------+-----------------+
|run_id       |batch_id |total_records|passed_records|failed_records|missing_customer_id_records|invalid_amount_records|duplicate_order_id_records|unknown_customer_records|pass_rate_percent|
+-------------+---------+-------------+--------------+--------------+---------------------------+----------------------+--------------------------+------------------------+-----------------+
|day21_run_001|batch_001|5            |1             |4             |1                          |1                     |0                         |1                       |20.0             |
|day21_run_001|batch_002|4            |1             |3             |0                          |1                     |2                         |1                       |25.0             |
+-------------+---------+-------------+------

## Quick recap

| Question | Answer |
|---|---|
| DQ check ต่างจาก cleansing ยังไง? | Cleansing ทำข้อมูลให้ standardized ส่วน DQ check ตรวจว่าข้อมูลผ่าน rule ที่คาดหวังหรือไม่ |
| ทำไมไม่ควร filter bad records ทิ้งเงียบ ๆ? | เพราะจะเสีย evidence ว่าข้อมูล fail เพราะอะไร และ debug ย้อนหลังยาก |
| `dq_status` ควรมาจากอะไร? | ควรมาจากผลรวมของ DQ flags / issue codes ไม่ใช่ hard-code แบบไม่มีเหตุผล |
| ทำไมต้องมี DQ summary? | เพื่อ monitor คุณภาพข้อมูลต่อ batch/run ได้เร็ว ไม่ต้องเปิดดู record ทุกแถว |
| Referential check ใช้ทำอะไร? | ตรวจว่า key ใน transaction/fact table มีอยู่ใน master/reference table จริงหรือไม่ |

## Exercises

### Exercise 1: Create batch-level required-field summary

สร้าง DataFrame ชื่อ `df_required_field_summary` จาก `df_dq_result` เพื่อสรุป required-field issues ต่อ `batch_id`

Requirements:

- group by `batch_id`
- count `total_records`
- sum `has_missing_order_id`
- sum `has_missing_customer_id`
- show result

Expected result:

- เห็นว่า batch ไหนมี required-field issue กี่ records
- ใช้ `.cast("int")` กับ boolean flags ก่อน sum

In [50]:
df_required_field_summary = (
    df_dq_result
    .groupBy("batch_id")
    .agg(
        F.count("*").alias("total_records"),
        F.sum(F.col("has_missing_order_id").cast("int")).alias("missing_order_id_records"),
        F.sum(F.col("has_missing_customer_id").cast("int")).alias("missing_customer_id_records"),
    )
    .orderBy("batch_id")
)

df_required_field_summary.show(truncate=False)

StatementMeta(, 29e3b4f8-e9a5-4ca9-8c69-0a6147e4b56c, 52, Finished, Available, Finished, False)

+---------+-------------+------------------------+---------------------------+
|batch_id |total_records|missing_order_id_records|missing_customer_id_records|
+---------+-------------+------------------------+---------------------------+
|batch_001|5            |0                       |1                          |
|batch_002|4            |0                       |0                          |
+---------+-------------+------------------------+---------------------------+



### Exercise 2: Create ready-only DataFrame for downstream jobs

สร้าง DataFrame ชื่อ `df_downstream_ready_orders` จาก `df_dq_result` สำหรับส่งต่อไป downstream transformation แบบง่าย ๆ

Requirements:

- filter เฉพาะ `dq_status = 'PASS'`
- select เฉพาะ business columns หลัก
- ไม่ต้อง write table เพิ่มใน exercise นี้
- ใช้ `.count()` และ `.show()` เพื่อตรวจผล

Expected result:

- ได้เฉพาะ records ที่ผ่าน DQ checks ทั้งหมด
- records ที่ fail required field, amount, date, duplicate, status หรือ referential check ยังอยู่ใน DQ result แต่ไม่ถูกเลือกเข้า `df_downstream_ready_orders`

In [51]:
df_downstream_ready_orders = (
    df_dq_result
    .filter(F.col("dq_status") == "PASS")
    .select(
        "order_id",
        "customer_id",
        "order_date",
        "amount",
        "order_status",
        "source_system",
        "batch_id",
    )
    .orderBy("order_id")
)

print("Downstream ready order count:", df_downstream_ready_orders.count())
df_downstream_ready_orders.show(truncate=False)

StatementMeta(, 29e3b4f8-e9a5-4ca9-8c69-0a6147e4b56c, 53, Finished, Available, Finished, False)

Downstream ready order count: 2
+--------+-----------+----------+------+------------+-------------+---------+
|order_id|customer_id|order_date|amount|order_status|source_system|batch_id |
+--------+-----------+----------+------+------------+-------------+---------+
|O001    |C001       |2026-05-01|1200.5|completed   |web          |batch_001|
|O008    |C002       |2026-05-07|450.0 |pending     |web          |batch_002|
+--------+-----------+----------+------+------------+-------------+---------+



### Exercise 3: Write exercise DQ summary table

สร้าง DataFrame ชื่อ `df_exercise_dq_status_summary` เพื่อสรุปจำนวน `PASS` / `FAIL` ต่อ `batch_id` และเขียนเป็น Lakehouse table ชื่อ `day21_exercise_dq_status_summary`

Requirements:

- group by `batch_id`, `dq_status`
- count records
- write table ด้วย `mode("overwrite")`
- read table กลับมาตรวจผล

Expected result:

- มี summary table ที่บอกแต่ละ batch มี PASS / FAIL กี่ records
- table สามารถอ่านกลับด้วย `spark.read.table()` ได้

In [52]:
exercise_summary_table_name = "day21_exercise_dq_status_summary"

df_exercise_dq_status_summary = (
    df_dq_result
    .groupBy("batch_id", "dq_status")
    .agg(F.count("*").alias("record_count"))
    .orderBy("batch_id", "dq_status")
)

df_exercise_dq_status_summary.show(truncate=False)

(
    df_exercise_dq_status_summary
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(exercise_summary_table_name)
)

spark.read.table(exercise_summary_table_name).show(truncate=False)

StatementMeta(, 29e3b4f8-e9a5-4ca9-8c69-0a6147e4b56c, 54, Finished, Available, Finished, False)

+---------+---------+------------+
|batch_id |dq_status|record_count|
+---------+---------+------------+
|batch_001|FAIL     |4           |
|batch_001|PASS     |1           |
|batch_002|FAIL     |3           |
|batch_002|PASS     |1           |
+---------+---------+------------+

+---------+---------+------------+
|batch_id |dq_status|record_count|
+---------+---------+------------+
|batch_001|FAIL     |4           |
|batch_001|PASS     |1           |
|batch_002|FAIL     |3           |
|batch_002|PASS     |1           |
+---------+---------+------------+



## Common mistakes

1. **Drop bad records ทิ้งโดยไม่มี log**

   ถ้า filter เฉพาะ valid records แล้วทิ้ง invalid records ไปเลย จะทำให้ไม่รู้ว่า source มีปัญหาอะไร และยากต่อการ audit ย้อนหลัง

2. **มีแค่ `is_valid` แต่ไม่มี reason**

   `is_valid = false` บอกแค่ว่า record ผิด แต่ไม่บอกว่าผิดเพราะอะไร ควรมี `dq_issues` หรือ issue code เพิ่มด้วย

3. **ทำ DQ หลัง aggregate อย่างเดียว**

   ถ้าตรวจหลัง aggregation อาจเสียรายละเอียดระดับ record ไปแล้ว ควรมี record-level checks ก่อนส่งต่อ downstream สำคัญ ๆ

4. **ใช้ business rule โดยไม่ normalize input ก่อน**

   เช่น status มีทั้ง `Completed`, ` completed `, `COMPLETED` ถ้าไม่ trim/lower ก่อน อาจถูก flag ว่า invalid ทั้งที่แก้ได้ง่าย

5. **ไม่แยก failed records ให้มองเห็น**

   DQ summary มีประโยชน์ แต่เวลา debug ต้องเห็น record-level failed data ด้วย ไม่ควรมีแค่ตัวเลขสรุปอย่างเดียว

## Expected Output / Success Criteria

เมื่อจบ Day 21 ควรทำได้:

- อธิบายความต่างระหว่าง data cleansing กับ data quality checks ได้
- สร้าง DQ flags สำหรับ required field, valid date, valid amount, valid status, uniqueness และ referential check ได้
- รวมหลาย DQ flags เป็น `dq_issues` และ `dq_status` ได้
- แยก ready records (`PASS`) และ failed records (`FAIL`) ได้โดยไม่ลบ failed records ทิ้งเงียบ ๆ
- สร้าง `df_dq_run_summary` เพื่อดู pass/fail ต่อ batch ได้
- สร้าง `df_dq_rule_summary` เพื่อดู failed records ต่อ rule code ได้

## Final takeaway

Data Quality Checks คือจุดที่ทำให้ pipeline ไม่ได้แค่ transform ข้อมูล แต่เริ่มบอกได้ว่าข้อมูลน่าเชื่อถือแค่ไหน

> Good DQ pattern should make data problems visible, explainable, and auditable.

จำ mindset หลักของวันนี้:

- สร้าง flags ก่อน แล้วค่อยรวมเป็น status
- อย่ามีแค่ fail/pass โดยไม่มี reason
- อย่า drop bad records ทิ้งโดยไม่มี evidence
- สรุปผล DQ ให้ดูได้ทั้ง record-level และ summary-level

## Optional cleanup

In [ ]:
# spark.sql("DROP TABLE IF EXISTS day21_dq_order_result")
# spark.sql("DROP TABLE IF EXISTS day21_dq_run_summary")
# spark.sql("DROP TABLE IF EXISTS day21_dq_rule_summary")
# spark.sql("DROP TABLE IF EXISTS day21_exercise_dq_status_summary")